# L01 – Arbetsflöde och miljö

**Tid:** ca 1,5 h · **Del A: Grunder**

## Mål
Efter lektionen ska du kunna:
1. beskriva det typiska arbetsflödet när man tränar ett neuralt nätverk,
2. installera PyTorch och välja `device` (GPU eller CPU),
3. sätta seeds för reproducerbarhet och förstå varför resultaten ändå kan variera lite,
4. dela upp data i **train/validation/test** och förklara varför man behöver tre delar.

> **Tips för code-along:** skriv helst av koden själv i stället för att bara köra cellerna. Det är när du skriver den som du märker vad du inte förstått än.

## 1. Arbetsflödet

Det här arbetsflödet återkommer i varje lektion. Du kan det mesta redan från scikit-learn. Det som är nytt i djupinlärning är främst steg 5–8.

| Steg | Vad | Motsvarighet i scikit-learn |
|---|---|---|
| 1 | Förstå problemet och välj **mått** (accuracy, F1, RMSE …) | samma |
| 2 | Dela upp i **train/val/test** | `train_test_split` |
| 3 | Ta fram en **baslinje**: naiv modell eller sklearn | `DummyClassifier`, `LogisticRegression` |
| 4 | Bygg en **datapipeline** med normalisering | `Pipeline` + `StandardScaler` |
| 5 | Gör en liten modell som kan **överanpassa en batch**, som sanity check | – |
| 6 | **Träna** och titta på inlärningskurvorna | `.fit()` (men här ser du varje epok) |
| 7 | **Diagnostisera**: över- eller underanpassning? | `learning_curve`, `validation_curve` |
| 8 | **Regularisera** och justera hyperparametrar mot val | `GridSearchCV`, `alpha`, `C` |
| 9 | Utvärdera **en gång** på test | `score` på testdata |
| 10 | **Spara** modellen | `joblib.dump` |

**Varför en baslinje?** Utan en baslinje vet du inte om 85 % accuracy är bra eller dåligt. Om en logistisk regression ger 84 % är ett neuralt nät med 85 % knappast värt den extra komplexiteten.

## 2. Installera PyTorch

Kör cellen nedan **en gång**. Se till att notebookens kernel är repots `.venv`. I VS Code väljer du det uppe till höger under *Select Kernel*.

**Varför ett särskilt index (`cu126`)?** PyTorch finns i olika byggen för olika CUDA-versioner. Din GTX 1080 är en *Pascal*-GPU med compute capability 6.1 (`sm_61`), och de nyaste byggena (CUDA 12.8+), som är de pip hämtar från PyPI som standard, har slutat stödja den. Därför pinnar `requirements.txt` den exakta versionen `torch==2.14.1+cu126` från PyTorchs eget index. Utan pinningen skulle pip välja den nyaste versionen från PyPI, och då skulle `torch.cuda.is_available()` ge `True` men varje beräkning på GPU:n misslyckas.

In [ ]:
%pip install -r ../requirements.txt

Starta om kerneln efter installationen (*Restart* i verktygsfältet). Kontrollera sedan versionerna och att GPU:n syns:

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA tillgängligt:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Arkitekturer i bygget:", torch.cuda.get_arch_list())  # sm_60 ska finnas med: den koden körs på alla Pascal-kort, även din sm_61

### Välj `device`

I PyTorch bestämmer du själv var beräkningarna görs. Data och modell måste ligga på **samma** device, annars får du ett fel.

- **GPU (`"cuda"`)**: tusentals små kärnor som gör matrisberäkningar parallellt. Det är ofta 10–50 gånger snabbare för neurala nät.
- **CPU (`"cpu"`)**: fungerar alltid och räcker för små modeller och tabelldata.

Den här raden används i början av varje notebook i kursen:

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

Vi gör ett litet test av skillnaden: en stor matrismultiplikation på CPU respektive GPU.

In [ ]:
import time

def time_matmul(device, n=4000, repeats=5):
    a = torch.rand(n, n, device=device)
    b = torch.rand(n, n, device=device)
    a @ b  # en första körning "värmer upp" (t.ex. laddar GPU-kärnor)
    if device == "cuda":
        torch.cuda.synchronize()  # GPU:n jobbar asynkront, vänta tills den är klar
    t0 = time.time()
    for _ in range(repeats):
        a @ b
    if device == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t0) / repeats

print(f"CPU: {time_matmul('cpu'):.3f} s per multiplikation")
if torch.cuda.is_available():
    print(f"GPU: {time_matmul('cuda'):.3f} s per multiplikation")

## 3. Reproducerbarhet med seeds

Ett neuralt nät är fullt av slump:
- vikterna initieras slumpmässigt,
- data blandas (shuffle) i varje epok,
- dropout (L07) slår av slumpmässiga neuroner.

Utan en fast **seed** får du olika resultat varje gång. Då blir det svårt att avgöra om en ändring faktiskt hjälpte eller om du bara hade tur. Det motsvarar `random_state=42` i scikit-learn, men här finns slumpgeneratorer på flera ställen och vi måste sätta alla:

In [ ]:
import random
import numpy as np

def set_seed(seed: int = 42) -> None:
    random.seed(seed)                 # Pythons inbyggda slump
    np.random.seed(seed)              # NumPy
    torch.manual_seed(seed)           # PyTorch på CPU
    torch.cuda.manual_seed_all(seed)  # PyTorch på alla GPU:er

set_seed(42)
print(torch.rand(3))
set_seed(42)
print(torch.rand(3))  # samma tal igen
print(torch.rand(3))  # nya tal: generatorn har gått vidare

**Varför kan resultaten ändå skilja sig lite?** Vissa GPU-operationer summerar tal i en ordning som inte är fast. Flyttalsaddition är inte exakt associativ, så `(a + b) + c` kan skilja sig från `a + (b + c)` i sista decimalen. Små avvikelser kan växa under träningen.

Det går att tvinga fram full determinism, men det gör träningen långsammare. Därför nöjer vi oss med seeds. **Praktisk slutsats:** skillnader på några tiondels procent mellan två körningar är brus, inte förbättringar.

## 4. Train, validation och test – varför tre delar?

Från scikit-learn känner du till train/test-split. I djupinlärning behöver vi nästan alltid **tre** delar:

| Del | Används till | Får påverka modellen? |
|---|---|---|
| **train** | att lära vikterna | ja, direkt |
| **validation** | att välja hyperparametrar och arkitektur, early stopping, att jämföra modeller | ja, *indirekt* genom dina val |
| **test** | slutlig uppskattning av hur bra modellen är på ny data | **nej**, används en gång i slutet |

**Varför räcker inte train/test?** Varje gång du tittar på testresultatet och ändrar något (fler lager, annan learning rate) anpassar *du* modellen till testdatan. Efter 50 sådana försök är testresultatet optimistiskt, eftersom testdatan i praktiken blivit träningsdata. Valideringsmängden tar den smällen, så att testmängden förblir opåverkad.

**Varför inte korsvalidering, som i sklearn?** Det går, men ett neuralt nät tar minuter eller timmar att träna. 5-faldig CV kostar fem gånger så mycket. Med mycket data (tiotusentals exempel) är dessutom en enda valideringsmängd tillräckligt stabil. CV används mest när datamängden är liten.

### Exempel: breast cancer-datan
Vi använder ett litet dataset från sklearn för att öva på arbetsflödet. Det finns inget neuralt nät än, bara uppdelningen och modellvalet.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
print("X:", X.shape, " y:", y.shape, " andel positiva:", y.mean().round(3))

# Steg 1: dela av test (20 %). Den rör vi inte förrän i slutet.
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
# Steg 2: dela resten i train (75 % av resten = 60 % totalt) och val (20 % totalt)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42
)
print("train:", X_train.shape, " val:", X_val.shape, " test:", X_test.shape)

`stratify=y` ger samma klassfördelning i alla tre delar. Det är viktigt när datamängden är liten eller obalanserad.

Nu följer vi steg 3 och 8 i arbetsflödet: en baslinje och sedan modellval **på valideringsmängden**.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Baslinje: gissa alltid den vanligaste klassen
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"Baslinje (vanligaste klassen), val accuracy: {dummy.score(X_val, y_val):.3f}")

# Modellval: jämför olika regulariseringsstyrka C på val
results = {}
for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
    model.fit(X_train, y_train)
    results[C] = model.score(X_val, y_val)
    print(f"C={C:<6} val accuracy: {results[C]:.3f}")

best_C = max(results, key=results.get)
print("Bästa C enligt val:", best_C)

Lägg märke till att `StandardScaler` ligger **i pipelinen**. Den anpassas bara på `X_train` och används sedan på val och test. Om du anpassar skalningen på all data läcker information från val och test in i träningen (*data leakage*). Samma princip gäller i PyTorch (L04).

Nu, och **bara nu**, utvärderar vi den valda modellen på test. Det är vanligt att träna om på train + val innan, eftersom mer data oftast ger en bättre modell.

In [ ]:
final_model = make_pipeline(StandardScaler(), LogisticRegression(C=best_C, max_iter=5000))
final_model.fit(X_trainval, y_trainval)
print(f"Test accuracy (en gång!): {final_model.score(X_test, y_test):.3f}")

## 5. Vad händer om … vi delar upp på ett annat sätt?

**Gissa först:** hur mycket varierar valideringsresultatet om vi bara byter `random_state` i uppdelningen?

In [ ]:
for seed in range(5):
    X_tr, X_v, y_tr, y_v = train_test_split(X_trainval, y_trainval, test_size=0.25,
                                            stratify=y_trainval, random_state=seed)
    model = make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=5000)).fit(X_tr, y_tr)
    print(f"seed {seed}: val accuracy {model.score(X_v, y_v):.3f}")

Valideringsmängden har bara cirka 114 exempel, så ett enda felklassat exempel ändrar accuracy med nästan 1 procentenhet. **Slutsats:** med liten valideringsmängd blir modellvalet osäkert. Då är korsvalidering värt kostnaden. Med 10 000 valideringsexempel (som i Fashion-MNIST senare) är bruset mycket mindre.

## 6. Övningar

### Övning 1 – en egen uppdelningsfunktion
Skriv `split_three` som delar `X, y` i train/val/test med de angivna andelarna (av *hela* datamängden), stratifierat och med fast seed.

*Ledtråd:* andelen i det andra anropet är `val_size / (1 - test_size)`.

In [ ]:
def split_three(X, y, val_size=0.2, test_size=0.2, seed=42):
    # TODO: använd train_test_split två gånger
    ...
    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
# Kör för att kontrollera
Xa, Xb, Xc, ya, yb, yc = split_three(X, y, val_size=0.15, test_size=0.15)
assert len(Xa) + len(Xb) + len(Xc) == len(X)
assert abs(len(Xc) / len(X) - 0.15) < 0.01, "testandelen ska vara ca 15 %"
assert abs(len(Xb) / len(X) - 0.15) < 0.01, "valideringsandelen ska vara ca 15 %"
assert abs(ya.mean() - yb.mean()) < 0.03, "glömde du stratify?"
print("Rätt!")

### Övning 2 – reproducerbarhet
Använd `set_seed` så att `a` och `b` nedan blir identiska och `c` blir annorlunda.

In [ ]:
# TODO: lägg in anrop till set_seed på rätt ställen
a = torch.randn(5)
b = torch.randn(5)
c = torch.randn(5)

In [ ]:
assert torch.equal(a, b), "a och b ska vara lika"
assert not torch.equal(b, c), "c ska skilja sig"
print("Rätt!")

### Övning 3 – reflektion (skriv svaret i cellen)
1. Du har testat 30 olika arkitekturer och valt den med bäst **test**-accuracy, 91 %. Varför är 91 % troligen en överskattning?
2. Varför är fast seed viktig när du jämför två learning rates?
3. När skulle du välja korsvalidering i stället för en fast valideringsmängd?

*Dina svar:*

1.
2.
3.

## När använder jag detta?
- **Train/val/test:** alltid. Testmängden används en gång, i slutet.
- **Korsvalidering:** vid små dataset (några tusen exempel eller färre) och när en träning tar kort tid.
- **Seeds:** alltid när du jämför experiment. Kör gärna 2–3 seeds om skillnaden är liten.
- **GPU:** för bilder, text och stora modeller. CPU räcker för små tabelldataset.

## Sammanfattning
- Arbetsflödet: mått, uppdelning, baslinje, pipeline, träning, diagnos, regularisering, test en gång och sparande.
- `device = "cuda" if torch.cuda.is_available() else "cpu"`, och data och modell på samma device.
- `set_seed` sätter slumpen för `random`, NumPy och PyTorch.
- Valideringsmängden skyddar testmängden från dina egna val.

**Nästa lektion:** tensorer och autograd, byggstenarna i PyTorch.